In [19]:
import os
os.chdir('/Users/jeffreybloodworth/Desktop/git-repos/scRNAseq-ExplainableML')
os.getcwd()

'/Users/jeffreybloodworth/Desktop/git-repos/scRNAseq-ExplainableML'

In [20]:
import pandas as pd
import numpy as np

counts = pd.read_csv(
    "data/processed/pdc_counts.csv.gz",
    index_col=0
)

meta = pd.read_csv(
    "data/processed/pdc_metadata.csv"
)

print(counts.shape)
print(meta.shape)

(16955, 4846)
(14430, 58)


In [21]:
meta.columns[:5]

Index(['cell', 'orig.ident', 'orig.ident2', 'nCount_RNA', 'nFeature_RNA'], dtype='str')

In [22]:
barcode_col = meta.columns[0]

all(
    meta[barcode_col].values
    ==
    counts.columns.values
)

ValueError: operands could not be broadcast together with shapes (14430,) (4846,) 

In [ ]:
gene_detection = (
    counts > 0
).sum(axis=1)

gene_detection.describe()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(
    gene_detection,
    bins=100
)

plt.xlabel("Cells expressing gene")
plt.ylabel("Number of genes")
plt.title("Gene Detection Frequency")
plt.show()

In [ ]:
plt.savefig('Gene Detection Frequency', dpi=300)

In [ ]:
filtered_counts = counts.loc[
    gene_detection >= 25
]

print(filtered_counts.shape)

In [ ]:
gene_var = filtered_counts.var(
    axis=1
)


In [ ]:
gene_var.describe()

In [ ]:
top_genes = (
    gene_var
    .sort_values(
        ascending=False
    )
    .head(2000)
    .index
)

ml_counts = filtered_counts.loc[
    top_genes
]

print(ml_counts.shape)

In [ ]:
X = ml_counts.T

In [ ]:
X.shape

In [ ]:
y = meta["is_malignant"]

In [ ]:
y.value_counts()

In [ ]:
X.to_csv(
    "data/processed/ml_matrix_top2000.csv.gz",
    compression="gzip"
)

In [ ]:
y.to_csv(
    "data/processed/ml_labels.csv",
    index=False
)

In [ ]:
## Feature Filtering Strategy

Genes detected in fewer than 25 pDCs were removed.

The remaining genes were ranked by variance across cells.

The top 2,000 most variable genes were retained for machine learning.

This reduces dimensionality while preserving biologically informative variation.

In [ ]:
X.shape

In [23]:
y.value_counts()

is_malignant
Malignant       4405
Premalignant     441
Name: count, dtype: int64

In [24]:
meta.columns[:5]

Index(['cell', 'orig.ident', 'orig.ident2', 'nCount_RNA', 'nFeature_RNA'], dtype='str')

In [25]:
print(X.shape)

print(meta["is_malignant"].value_counts())

(4846, 2000)
is_malignant
Malignant       13732
Premalignant      495
Healthy           203
Name: count, dtype: int64
